# People's Speech Annotation Pipeline

Runs **PSST** and **Wav2ToBI** on People's Speech (MLCommons, clean/CC-BY subset) samples,
computes consensus phrase boundaries, classifies F0 intonation direction at boundaries,
and writes structured batch label files to Google Drive.

This is a direct port of the LibriTTS annotation pipeline (`annotation_pipeline_psst_w2t.ipynb`)
with three changes: (1) data source is `MLCommons/peoples_speech` config `"clean"`,
(2) text field is `text` instead of `text_normalized`, and (3) an hour-cap mechanism
limits the master ID list to ~1,000–2,000 h to stay tractable on Colab T4.

Disclaimer: Code was largely generated with the help of Claude Sonnet 4.6 (Anthropic, 2026 June).
Prompts, code tweaks and verification by me.

---

## How to use this notebook

Run cells **top to bottom**. The only cell you should need to edit is **Cell 1 (Configuration)**.

| Section | Cells | What it does |
|---|---|---|
| **1 · Configuration** | 1–2 | Set paths, hyperparameters, run mode |
| **2 · Environment** | 3–5 | Mount Drive, install packages, clone & patch Wav2ToBI |
| **3 · Load Models** | 6–7 | Load PSST (Whisper) and Wav2ToBI (Wav2Vec2) |
| **4 · Helper Functions** | 8–12 | Define all processing logic (no edits needed) |
| **5 · Data Loading** | 13 | Stream People's Speech, run CTC forced alignment |
| **6 · Run Pipeline** | 14 | Process samples, write label files |
| **7 · Analysis** | 15 | Summary statistics and agreement report |

---

## Output folder structure

```
labels/peoples_speech/
├── meta.json
├── batch_0000.json
├── batch_0001.json
└── ...
```

Each batch file is a dict keyed by sample ID, each value has `b` / `i` / `x` keys.
Schema is identical to the LibriTTS batched labels — drop-in compatible with the DistilBERT loader.

---

## Key design decisions

- **Data source:** `MLCommons/peoples_speech`, config `"clean"` (CC-BY licensed).
  Audio is read speech at 16 kHz. Transcripts already exist — no Whisper needed.
- **Hour cap:** `MAX_HOURS` in Cell 1 limits the master ID list. 1,500 h is the default.
- **Cache layout:** `cache/peoples_speech/` is separate from the LibriTTS cache
  (`cache/`) to avoid ID collisions between corpora.
- **Resume safety:** processed IDs are tracked in `progress_peoples_speech.json`;
  re-running Cell 13 skips already-annotated samples automatically.
- **Annotation logic** (boundaries, intonation, break index, consensus) is identical
  to the LibriTTS pipeline. All helper functions (Cells 6–11) are unchanged.


---
## Section 1 · Configuration

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — CONFIGURATION                                      ║
# ║  This is the only cell you need to edit between runs.        ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Sampling ─────────────────────────────────────────────────────
# MODE = "chunk"    → process up to N_SAMPLES samples per run (resume-safe)
# MODE = "specific" → process only SPECIFIC_IDS
MODE            = "chunk"
N_SAMPLES       = 31845
SPECIFIC_IDS    = []
RANDOM_SEED     = 42

# ── Hour cap ──────────────────────────────────────────────────────
# Stop adding samples to the master ID list once this many hours of
# audio have been accumulated.  Set to None to disable.
# People's Speech clean subset is ~3,000 h
MAX_HOURS       = 3000

# ── Paths ────────────────────────────────────────────────────────
DRIVE_ROOT     = "/content/drive/MyDrive/Capstone/project"
CACHE_ROOT     = f"{DRIVE_ROOT}/cache/peoples_speech"

AUDIO_CACHE    = f"{CACHE_ROOT}/cache/ps_audio"  # cached .wav files (optional)
ANNOT_CACHE    = f"{CACHE_ROOT}/annotations"     # full provenance JSONs
CTC_CACHE      = f"{CACHE_ROOT}/ctc"       # word timestamps per sample
PSST_CACHE_DIR = f"{CACHE_ROOT}/psst"      # raw PSST decoded strings
W2T_CACHE_DIR  = f"{CACHE_ROOT}/w2t"       # raw Wav2ToBI detections

WAV2TOBI_DIR   = f"{DRIVE_ROOT}/wav2tobi"

# ── Label output path ─────────────────────────────────────────────
BATCHED_ROOT   = f"{DRIVE_ROOT}/labels/ps"
MASTER_ID_FILE = f"{CACHE_ROOT}/master_ids_peoples_speech.json"
PROGRESS_FILE  = f"{CACHE_ROOT}/progress_peoples_speech.json"

# ── Run mode ─────────────────────────────────────────────────────
# "full"       → PSST + Wav2ToBI + merge in one pass (original single-session)
# "psst_only"  → Session A: CTC alignment + PSST only; saves ctc_cache + psst_cache
# "w2t_only"   → Session B: reads ctc_cache; Wav2ToBI only; saves w2t_cache
# "merge_only" → reads psst_cache + w2t_cache + ctc_cache; writes label files (no GPU)
#
# Two-session workflow:
#   Session A:  RUN_MODE = "psst_only"   (loads PSST + CTC)
#   Session B:  RUN_MODE = "w2t_only"    (loads Wav2ToBI only)
#   Either:     RUN_MODE = "merge_only"  (no model needed)
RUN_MODE = "full"

# ── Model checkpoints ────────────────────────────────────────────
PSST_CKPT     = "NathanRoll/psst-medium-en"
WAV2TOBI_CKPT = "ReginaZ/Wav2ToBI-PB-Fuzzy"

# ── Batching ─────────────────────────────────────────────────────
# How many audio samples to feed PSST in one forward pass.
# 4 is safe on a T4 (15 GB VRAM). Increase to 6-8 if you have headroom.
PSST_BATCH_SIZE = 4

# ── Hyperparameters ──────────────────────────────────────────────
# Copied from LibriTTS pipeline (empirically validated on read speech).
# People's Speech clean subset is also read speech, so these values
# should transfer well.
ALIGNMENT_TOLERANCE_S = 0.15
LAST_WORD_LATE_TOLERANCE_S  = 0.25
LAST_WORD_EARLY_TOLERANCE_S = 0.50
THRESHOLD_MULT    = 0.5
HIGH_MULT         = 2.5
F0_WINDOW_S       = 0.20
F0_SLOPE_HZ       = 2.0
CONSENSUS_WINDOW  = 1

# ── Processing control ───────────────────────────────────────────
FORCE_REPROCESS = False

# ── Audio storage ─────────────────────────────────────────────────
# People's Speech audio is already at 16 kHz — no resampling needed
# in most cases.  KEEP_AUDIO = True caches .wav to AUDIO_CACHE so
# re-runs don't need to re-stream.  Costs ~50 MB/h of audio on Drive.
# False is recommended for >500h runs to save Drive quota.
KEEP_AUDIO = False

print("✓ Configuration loaded.")
print(f"  RUN_MODE:       {RUN_MODE}")
print(f"  Mode:           {MODE}  |  Samples: {N_SAMPLES if MODE != 'specific' else len(SPECIFIC_IDS)}")
print(f"  MAX_HOURS:      {MAX_HOURS}")
print(f"  KEEP_AUDIO:     {KEEP_AUDIO}")
print(f"  Batch size (PSST): {PSST_BATCH_SIZE}")
print(f"  Drive root:     {DRIVE_ROOT}")
print(f"  Batched labels: {BATCHED_ROOT}")

---
## Section 2 · Environment Setup

Run once per Colab session. Safe to re-run — all steps are idempotent.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Mount Drive + install packages                     ║
# ╚══════════════════════════════════════════════════════════════╝
from google.colab import drive
import os

drive.mount("/content/drive", force_remount=True)

# Create all required directories
for d in [AUDIO_CACHE, ANNOT_CACHE,
          CTC_CACHE, PSST_CACHE_DIR, W2T_CACHE_DIR,
          BATCHED_ROOT]:
    os.makedirs(d, exist_ok=True)

print("✓ Drive mounted. All directories created.")

# Install packages not pre-installed on Colab
# (praat-parselmouth, soundfile, evaluate, librosa are not bundled)
!pip install -q praat-parselmouth soundfile evaluate librosa

import torch
print(f"\n✓ PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("⚠  WARNING: No GPU detected. PSST inference will be very slow (~1 min/sample).")
    print("   Runtime → Change runtime type → T4 GPU")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"   Device: {device}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2b — Load / initialize progress ledger                ║
# ╚══════════════════════════════════════════════════════════════╝
import json, os

if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE) as f:
        completed_ids = set(json.load(f))
    print(f"✓ Ledger loaded: {len(completed_ids)} samples already completed.")
else:
    completed_ids = set()
    print("✓ No ledger found — starting fresh.")

---
## Section 2c · Master ID List

Run **once** to build `master_ids_peoples_speech.json`.
Streams People's Speech metadata (IDs + `duration_ms` only — no audio at this stage)
until the `MAX_HOURS` cap is reached, then saves the ID list to Drive.
On subsequent sessions the file is loaded directly — no re-streaming needed.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2c — Build master ID list from People's Speech         ║
# ║                                                              ║
# ║  Streams MLCommons/peoples_speech (clean, CC-BY subset)      ║
# ║  and collects sample IDs until MAX_HOURS of audio have been  ║
# ║  seen.  IDs are saved to MASTER_ID_FILE and reused on        ║
# ║  subsequent runs — the stream is never re-traversed.         ║
# ║                                                              ║
# ║  Each People's Speech sample has:                            ║
# ║    id              — string (e.g. "common_voice_en_23616981") ║
# ║    audio           — dict with array (float32) + sampling_rate║
# ║    duration_ms     — audio duration in milliseconds          ║
# ║    text            — transcript (may be ASR-generated for    ║
# ║                      some clean samples, but quality is high) ║
# ╚══════════════════════════════════════════════════════════════╝
import os, json
from datasets import load_dataset

if os.path.exists(MASTER_ID_FILE):
    with open(MASTER_ID_FILE) as f:
        master_ids = json.load(f)
    print(f"✓ Master ID list loaded: {len(master_ids)} samples")
    meta_path = MASTER_ID_FILE.replace(".json", "_meta.json")
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            id_meta = json.load(f)
        total_h = id_meta.get("total_hours", "?")
        if isinstance(total_h, float):
            print(f"  Approximate audio: {total_h:.1f} h")
        else:
            print(f"  Hours: {total_h}")
    print(f"  First 3: {master_ids[:3]}")

else:
    print(f"Building master ID list from MLCommons/peoples_speech (clean)...")
    print(f"  Hour cap: {MAX_HOURS} h")
    print()

    ds = load_dataset(
        "MLCommons/peoples_speech",
        "clean",
        split="train",
        streaming=True,
        trust_remote_code=True,
    )

    master_ids = []
    total_ms   = 0.0
    target_ms  = (MAX_HOURS * 3600 * 1000) if MAX_HOURS is not None else float("inf")
    n_seen     = 0

    for sample in ds:
        dur_ms = sample.get("duration_ms")
        if dur_ms is None:
            audio = sample.get("audio")
            if audio and "array" in audio and "sampling_rate" in audio:
                dur_ms = len(audio["array"]) / audio["sampling_rate"] * 1000
            else:
                dur_ms = 0.0

        master_ids.append(sample["id"])
        total_ms += dur_ms
        n_seen   += 1

        if n_seen % 10000 == 0:
            print(f"  Seen {n_seen:,} samples  ({total_ms/3_600_000:.1f} h accumulated)...")

        if total_ms >= target_ms:
            print(f"  Hour cap reached at sample {n_seen:,}.")
            break

    total_h = total_ms / 3_600_000
    print(f"\n✓ Collected {len(master_ids):,} sample IDs ({total_h:.1f} h)")

    with open(MASTER_ID_FILE, "w") as f:
        json.dump(master_ids, f)

    meta_path = MASTER_ID_FILE.replace(".json", "_meta.json")
    with open(meta_path, "w") as f:
        json.dump({"total_hours": total_h, "n_samples": len(master_ids),
                   "max_hours_cap": MAX_HOURS,
                   "corpus": "MLCommons/peoples_speech/clean"}, f, indent=2)

    print(f"  Saved → {MASTER_ID_FILE}")
    print(f"  First 3: {master_ids[:3]}")
    print(f"  Last  3: {master_ids[-3:]}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Clone and patch Wav2ToBI                          ║
# ║                                                              ║
# ║  Two patches are required every session:                     ║
# ║  1. train.py: replace deprecated `load_metric` import       ║
# ║  2. model.py: set lstm_hidden_size = 256 for PB checkpoints ║
# ║     (PA checkpoints need 128 — edit target below if needed) ║
# ╚══════════════════════════════════════════════════════════════╝
import os, re

# ── Clone (skip if already present) ─────────────────────────────
if not os.path.exists(WAV2TOBI_DIR):
    print("Cloning Wav2ToBI repository...")
    !git clone https://github.com/reginazhai/Wav2ToBI.git {WAV2TOBI_DIR}
    !pip install -q -r {WAV2TOBI_DIR}/requirements.txt
else:
    print("Wav2ToBI already cloned — applying patches.")

# ── Patch 1: fix deprecated load_metric import ──────────────────
TRAIN_PY = f"{WAV2TOBI_DIR}/src/train.py"
with open(TRAIN_PY) as f:
    src = f.read()
if "from datasets import load_metric" in src:
    src = src.replace(
        "from datasets import load_metric",
        "from evaluate import load as load_metric"
    )
    with open(TRAIN_PY, "w") as f:
        f.write(src)
    print("✓ Patch 1 applied: load_metric import fixed in train.py")
else:
    print("✓ Patch 1 already applied")

# ── Patch 2: set correct LSTM hidden size ────────────────────────
# PB checkpoints (Wav2ToBI-PB-Fuzzy / Wav2ToBI-PB-Flat) → 256
# PA checkpoints (Wav2ToBI-PA-Fuzzy / Wav2ToBI-PA-Flat) → 128
TARGET_HIDDEN_SIZE = 256   # ← change to 128 if loading a PA checkpoint

MODEL_PY = f"{WAV2TOBI_DIR}/src/model.py"
with open(MODEL_PY) as f:
    src = f.read()
match = re.search(r'self\.lstm_hidden_size = (\d+)', src)
if match:
    current = int(match.group(1))
    if current != TARGET_HIDDEN_SIZE:
        src = re.sub(r'self\.lstm_hidden_size = \d+',
                     f'self.lstm_hidden_size = {TARGET_HIDDEN_SIZE}', src)
        with open(MODEL_PY, "w") as f:
            f.write(src)
        print(f"✓ Patch 2 applied: lstm_hidden_size {current} → {TARGET_HIDDEN_SIZE}")
    else:
        print(f"✓ Patch 2 already applied (lstm_hidden_size = {TARGET_HIDDEN_SIZE})")

print("\n✓ Wav2ToBI ready.")


---
## Section 3 · Load Models

Both models are loaded once per session. Loading is skipped for models
not needed by the current `RUN_MODE`.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Load PSST                                          ║
# ╚══════════════════════════════════════════════════════════════╝
if RUN_MODE in ("full", "psst_only"):
    from transformers import WhisperProcessor, WhisperForConditionalGeneration
    print(f"Loading PSST from {PSST_CKPT}...")
    psst_processor = WhisperProcessor.from_pretrained(PSST_CKPT)
    psst_model     = WhisperForConditionalGeneration.from_pretrained(PSST_CKPT).to(device)
    psst_model.eval()
    print("✓ PSST loaded.")
else:
    psst_processor = None
    psst_model     = None
    print(f"Skipping PSST model load (RUN_MODE='{RUN_MODE}').")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Load Wav2ToBI + CTC model                         ║
# ╚══════════════════════════════════════════════════════════════╝
import sys, importlib

if RUN_MODE in ("full", "w2t_only"):
    from transformers import Wav2Vec2FeatureExtractor
    W2T_SRC = f"{WAV2TOBI_DIR}/src"
    if W2T_SRC not in sys.path:
        sys.path.insert(0, W2T_SRC)
    import model as _w2t_module
    importlib.reload(_w2t_module)
    from model import Wav2Vec2ForAudioFrameClassification_custom

    print(f"Loading Wav2ToBI from {WAV2TOBI_CKPT}...")
    w2t_processor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-base")
    w2t_model     = Wav2Vec2ForAudioFrameClassification_custom.from_pretrained(
        WAV2TOBI_CKPT, ignore_mismatched_sizes=True).to(device)
    w2t_model.eval()
    print(f"✓ Wav2ToBI loaded. lstm_hidden_size = {w2t_model.lstm_hidden_size}")
else:
    w2t_processor = None
    w2t_model     = None
    print(f"Skipping Wav2ToBI model load (RUN_MODE='{RUN_MODE}').")

# CTC forced alignment — needed in all modes except merge_only
# (merge_only reads mfa_words from ctc_cache on disk)
if RUN_MODE != "merge_only":
    import torchaudio
    print("Loading CTC alignment model...")
    FA_BUNDLE   = torchaudio.pipelines.WAV2VEC2_ASR_BASE_960H
    fa_model    = FA_BUNDLE.get_model().to(device)
    fa_labels   = FA_BUNDLE.get_labels()
    fa_model.eval()
    FA_STRIDE_S = 320 / 16000
    print("✓ CTC model loaded.")
else:
    fa_model    = None
    fa_labels   = None
    FA_STRIDE_S = 320 / 16000
    print("Skipping CTC model load (merge_only reads from ctc_cache).")


---
## Section 4 · Helper Functions

These cells define all processing logic. No edits needed — identical to
the LibriTTS pipeline.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — PSST inference helpers                            ║
# ╚══════════════════════════════════════════════════════════════╝
import re
import torch
import numpy as np
import librosa
from difflib import SequenceMatcher

BOUNDARY_TOKEN = "!!!!!"


def _psst_max_new_tokens(audio_array, sr=16000):
    # Whisper outputs ~5-8 tokens/sec of speech (words + boundary markers).
    # 10 tokens/sec is a generous upper bound; floor=30, cap=444.
    duration_s = len(audio_array) / sr
    return min(444, max(30, int(duration_s * 10)))


def run_psst(audio_array, sr=16000):
    """
    Run PSST on a single 16 kHz numpy audio array.
    Returns the raw decoded string, e.g. "The moon !!!!! I gazed with a kind of wonder".
    """
    if sr != 16000:
        audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=16000)

    inputs = psst_processor(
        audio_array, sampling_rate=16000, return_tensors="pt"
    ).to(device)

    tokenizer = psst_processor.tokenizer
    prefix_ids = [
        tokenizer.convert_tokens_to_ids("<|startoftranscript|>"),
        tokenizer.convert_tokens_to_ids("<|en|>"),
        tokenizer.convert_tokens_to_ids("<|transcribe|>"),
        tokenizer.convert_tokens_to_ids("<|notimestamps|>"),
    ]
    decoder_input_ids = torch.tensor([prefix_ids], device=device)
    max_new_tokens    = _psst_max_new_tokens(audio_array)

    with torch.no_grad():
        generated = psst_model.generate(
            inputs.input_features,
            decoder_input_ids=decoder_input_ids,
            suppress_tokens=[],
            max_new_tokens=max_new_tokens,
        )
    return psst_processor.batch_decode(generated, skip_special_tokens=False)[0]


def run_psst_batch(audio_arrays, sr=16000):
    """
    Run PSST on a list of audio arrays in one batched forward pass.

    Returns a list of raw decoded strings, one per input.

    Whisper's feature extractor always pads/truncates to 30 s, so all
    items in the batch have identical encoder-input shapes — no custom
    collation needed. Typical speedup vs single-sample: 2-4x on T4.

    Safe batch sizes on T4 (15 GB VRAM):  4-8  (psst-medium-en ~1.5 GB).
    """
    resampled = []
    for arr in audio_arrays:
        if sr != 16000:
            arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
        resampled.append(arr)

    inputs = psst_processor(
        resampled, sampling_rate=16000, return_tensors="pt", padding=True
    ).to(device)

    tokenizer  = psst_processor.tokenizer
    prefix_ids = [
        tokenizer.convert_tokens_to_ids("<|startoftranscript|>"),
        tokenizer.convert_tokens_to_ids("<|en|>"),
        tokenizer.convert_tokens_to_ids("<|transcribe|>"),
        tokenizer.convert_tokens_to_ids("<|notimestamps|>"),
    ]
    batch_size        = inputs.input_features.shape[0]
    decoder_input_ids = torch.tensor([prefix_ids] * batch_size, device=device)
    # Token cap based on the longest sample in the batch
    max_dur        = max(len(a) / 16000 for a in resampled)
    max_new_tokens = min(444, max(30, int(max_dur * 10)))

    with torch.no_grad():
        generated = psst_model.generate(
            inputs.input_features,
            decoder_input_ids=decoder_input_ids,
            suppress_tokens=[],
            max_new_tokens=max_new_tokens,
        )
    return psst_processor.batch_decode(generated, skip_special_tokens=False)


def parse_psst_output(psst_text):
    text = psst_text
    for tok in ["<|startoftranscript|>", "<|transcribe|>", "<|en|>",
                "<|notimestamps|>", "<|endoftext|>"]:
        text = text.replace(tok, "")
    text = text.strip()
    segments = text.split(BOUNDARY_TOKEN)
    result = []
    for i, seg in enumerate(segments):
        words = [w for w in seg.strip().split() if w]
        has_boundary = (i < len(segments) - 1)
        result.append((words, has_boundary))
    return result


def align_word_lists(psst_words, mfa_words):
    def normalize(w):
        return re.sub(r"[^a-z']", "", w.lower())
    psst_norm = [normalize(w) for w in psst_words]
    mfa_norm  = [normalize(w) for w in mfa_words]
    matcher = SequenceMatcher(None, psst_norm, mfa_norm, autojunk=False)
    mapping = {}
    for op, i1, i2, j1, j2 in matcher.get_opcodes():
        if op == 'equal':
            for k in range(i2 - i1):
                mapping[i1 + k] = j1 + k
        elif len(range(i1, i2)) > 0 and len(range(j1, j2)) > 0:
            for k, pi in enumerate(range(i1, i2)):
                mi = j1 + round(k / max(i2 - i1 - 1, 1) * max(j2 - j1 - 1, 0))
                mapping[pi] = min(mi, j2 - 1)
    return mapping


def psst_to_word_boundaries(psst_text, mfa_words):
    segments = parse_psst_output(psst_text)
    all_psst_words = []
    boundary_psst_indices = []
    for (words, has_boundary) in segments:
        all_psst_words.extend(words)
        if has_boundary and words:
            boundary_psst_indices.append(len(all_psst_words) - 1)
    if not all_psst_words:
        return []
    mfa_word_strs = [w['word'] for w in mfa_words]
    mapping = align_word_lists(all_psst_words, mfa_word_strs)
    boundaries = []
    for pi in boundary_psst_indices:
        if pi in mapping:
            boundaries.append(mapping[pi])
        else:
            prop = pi / max(len(all_psst_words) - 1, 1)
            boundaries.append(round(prop * (len(mfa_words) - 1)))
    return sorted(set(boundaries))


print("✓ Cell 6: PSST helpers defined (single + batched inference, dynamic max_new_tokens).")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Wav2ToBI inference helpers                        ║
# ╚══════════════════════════════════════════════════════════════╝
import torch
import parselmouth
import numpy as np
import librosa
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d


def run_wav2tobi(audio_array, sr=16000):
    """
    Run Wav2ToBI (PB checkpoint) on a numpy audio array.

    How it works:
    1. Extract F0 using Parselmouth (Praat). Wav2ToBI was trained with F0
       appended to the Wav2Vec2 frame embeddings, so we must replicate this.
    2. Run the Wav2Vec2 + BLSTM forward pass to get per-frame logits.
    3. Smooth the logit curve with a Gaussian filter.
    4. Detect peaks above an adaptive threshold (mean + threshold_mult * std).
       This avoids hard-coding a threshold that depends on the logit scale.
    5. Classify each peak as break index "4" (intonational phrase, higher
       logit) or "3" (intermediate phrase, lower logit) using the high mark.

    Returns list of [timestamp_seconds, break_index_string] pairs.
    """
    if sr != 16000:
        audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=16000)
        sr = 16000

    # Step 1: Extract F0
    snd = parselmouth.Sound(audio_array, sr)
    pitch_obj = snd.to_pitch(time_step=0.02, pitch_floor=75, pitch_ceiling=500)
    pitch_values = pitch_obj.selected_array['frequency']
    pitch_tensor = torch.tensor(pitch_values, dtype=torch.float32)

    # Step 2: Forward pass
    inputs = w2t_processor(audio_array, sampling_rate=sr, return_tensors="pt", padding=True)
    input_values = inputs.input_values.to(device)
    with torch.no_grad():
        outputs = w2t_model(input_values, pitch=[pitch_tensor])

    logits = outputs.logits.squeeze().cpu().numpy()
    if logits.ndim > 1:
        logits = logits[:, 0]

    # Diagnostic: print logit range so threshold decisions are visible
    print(f"    [W2T logits]  min={logits.min():.3f}  max={logits.max():.3f}  "
          f"mean={logits.mean():.3f}  std={logits.std():.3f}")

    # Step 3: Smooth
    smoothed = gaussian_filter1d(logits.astype(float), sigma=3)

    # Step 4: Adaptive threshold
    threshold = max(logits.mean() + THRESHOLD_MULT * logits.std(), 0.1)
    print(f"    [W2T threshold] {threshold:.3f}")

    # Wav2Vec2 temporal stride: 320 samples @ 16 kHz = 20 ms per frame
    FRAME_S = 320 / sr

    peaks, _ = find_peaks(
        smoothed,
        height=threshold,
        distance=int(0.2 / FRAME_S)   # minimum 200ms between peaks
    )

    # Step 5: Classify each peak as break index 3 or 4
    high_thresh = logits.mean() + HIGH_MULT * logits.std()
    detections = []
    for idx in peaks:
        t = round(idx * FRAME_S, 3)
        break_idx = "4" if smoothed[idx] >= high_thresh else "3"
        detections.append([t, break_idx])

    return detections


def wav2tobi_to_word_boundaries(detections, mfa_words, tolerance_s=ALIGNMENT_TOLERANCE_S):
    """
    Map Wav2ToBI timestamp detections to MFA word indices.

    For each detected boundary timestamp, find the word whose end time is
    closest. Acceptance rules:
      • If the closest word is ANY non-final word: accept iff |gap| ≤ tolerance_s
        (the standard strict 0.15s rule).
      • If the closest word is the UTTERANCE-FINAL word: accept under a wider
        asymmetric tolerance (LAST_WORD_EARLY_TOLERANCE_S / LAST_WORD_LATE_TOLERANCE_S).
        Empirically, W2T's final-word peak frequently drifts past 0.15s —
        the corpus diagnostic (N=13,291) showed late-side p99 = 0.18s and
        early-side p99 = 0.45s. The caps below encompass ~99% of legitimate
        drift while rejecting garbage (next-utterance leak-in late; internal
        prosody of hyphenated compound tokens early).

    Rejected peaks are logged and discarded.
    Returns list of (word_index, break_index_string) pairs.
    """
    results = []
    if not mfa_words:
        return results
    last_idx = len(mfa_words) - 1
    last_end = mfa_words[last_idx]['end']

    for (timestamp, break_idx) in detections:
        best_idx, best_dist = None, float('inf')
        for i, w in enumerate(mfa_words):
            dist = abs(w['end'] - timestamp)
            if dist < best_dist:
                best_dist = dist
                best_idx = i
        if best_idx is None:
            continue

        # Choose the applicable tolerance for this detection.
        if best_idx == last_idx:
            signed_gap = timestamp - last_end        # neg = early, pos = late
            if signed_gap >= 0:
                accepted = signed_gap <= LAST_WORD_LATE_TOLERANCE_S
                tol_tag  = f"final-late≤{LAST_WORD_LATE_TOLERANCE_S}s"
            else:
                accepted = (-signed_gap) <= LAST_WORD_EARLY_TOLERANCE_S
                tol_tag  = f"final-early≤{LAST_WORD_EARLY_TOLERANCE_S}s"
        else:
            accepted = best_dist <= tolerance_s
            tol_tag  = f"strict≤{tolerance_s}s"

        if accepted:
            results.append((best_idx, break_idx))
        else:
            print(f"    ⚠ W2T peak at {timestamp:.2f}s: "
                  f"closest gap {best_dist:.2f}s exceeds {tol_tag} — discarded")

    return results


print("✓ Cell 7: Wav2ToBI helpers defined.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Consensus and agreement helpers                   ║
# ╚══════════════════════════════════════════════════════════════╝


def compute_consensus(psst_indices, w2t_pairs, window=CONSENSUS_WINDOW):
    """
    Find word indices where PSST and Wav2ToBI agree within `window` words.

    PSST boundary positions are used as canonical (they are already aligned
    to the transcript via sequence matching). We accept a Wav2ToBI boundary
    at word i as matching a PSST boundary at word j if |i - j| <= window.

    Returns sorted list of agreed word indices.
    """
    psst_set = set(psst_indices)
    w2t_set  = set(idx for idx, _ in w2t_pairs)

    consensus = set()
    for pi in psst_set:
        if any(abs(pi - wi) <= window for wi in w2t_set):
            consensus.add(pi)

    return sorted(consensus)


def classify_agreement(psst_indices, w2t_indices, n_words, window=CONSENSUS_WINDOW):
    """ Classify each word position as one of four agreement statuses:
      - "agree_boundary"  : both annotators placed a boundary here
      - "agree_none"      : both annotators placed no boundary here
      - "psst_only"       : PSST fired, Wav2ToBI did not
      - "wav2tobi_only"   : Wav2ToBI fired, PSST did not
      Returns list of status strings aligned to word positions 0..n_words-1."""
    psst_set = set(psst_indices)
    w2t_set  = set(w2t_indices)

    psst_expanded = {pi + d for pi in psst_set for d in range(-window, window + 1)}
    w2t_expanded  = {wi + d for wi in w2t_set  for d in range(-window, window + 1)}

    status = []
    for i in range(n_words):
        in_psst = i in psst_set
        in_w2t  = i in w2t_set
        if in_psst and i in w2t_expanded:
            status.append("agree_boundary")
        elif in_psst:                          # PSST fired, no W2T nearby
            status.append("psst_only")
        elif in_w2t and i not in psst_expanded: # W2T fired, no PSST nearby
            status.append("wav2tobi_only")
        else:
            status.append("agree_none")         # no boundary OR W2T near PSST (already counted)
    return status


print("✓ Cell 8: Consensus and agreement helpers defined.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 9 — F0 intonation direction classifier                ║
# ║                                                              ║
# ║  This is a rule-based heuristic using Parselmouth/Praat.    ║
# ║  It measures F0 slope in the last F0_WINDOW_S seconds of a  ║
# ║  boundary word, then classifies as rising/falling/level.    ║
# ║                                                              ║
# ║  NOTE: This is a single-source label — there is no second   ║
# ║  annotator to compare against. It is an approximation of    ║
# ║  ToBI boundary tone (H-% ≈ rising, L-% ≈ falling) but is   ║
# ║  NOT equivalent to pitch accent detection.                  ║
# ╚══════════════════════════════════════════════════════════════╝
import parselmouth
import numpy as np


def classify_intonation(audio_array, sr, word_end_s):
    """
    Classify the F0 direction at a phrase boundary.

    Extracts F0 from the last F0_WINDOW_S seconds before `word_end_s`,
    computes the slope between the first and second half of voiced frames,
    and returns one of: 'rising' | 'falling' | 'level' | 'unclear'.

    'unclear' is returned when there are fewer than 4 voiced frames in
    the window (e.g. the boundary word is unvoiced or very short).
    """
    start_s = max(0.0, word_end_s - F0_WINDOW_S)
    segment = audio_array[int(start_s * sr) : int(word_end_s * sr)]

    # Require at least 50ms of audio to attempt classification
    if len(segment) < int(0.05 * sr):
        return 'unclear'

    try:
        snd = parselmouth.Sound(segment, sr)
        pitch = snd.to_pitch(time_step=0.01, pitch_floor=75, pitch_ceiling=500)
        f0 = pitch.selected_array['frequency']
        f0_voiced = f0[f0 > 0]
    except Exception:
        return 'unclear'

    if len(f0_voiced) < 4:
        return 'unclear'

    # Compare mean F0 in the first half vs. second half of the voiced segment
    mid = len(f0_voiced) // 2
    slope = np.mean(f0_voiced[mid:]) - np.mean(f0_voiced[:mid])

    if slope >  F0_SLOPE_HZ:  return 'rising'
    if slope < -F0_SLOPE_HZ:  return 'falling'
    return 'level'


print("✓ Cell 9: F0 intonation classifier defined.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Label file I/O                                   ║
# ║                                                              ║
# ║  Writes per-feature label files to the labels/ folder.      ║
# ║  Also handles the raw annotation cache (full provenance)    ║
# ║  and the three new per-annotator raw caches:                ║
# ║    ctc_cache/   — word timestamps (mfa_words)               ║
# ║    psst_cache/  — PSST raw decoded strings                  ║
# ║    w2t_cache/   — Wav2ToBI raw detections                   ║
# ╚══════════════════════════════════════════════════════════════╝

import json, os
from datetime import datetime
import time

INTONATION_MAP = {'none': 0, 'rising': 1, 'falling': 2, 'level': 3, 'unclear': 4}
FILE_BATCH_SIZE = 1000   # samples per batch file

# ── Batch buffer state ────────────────────────────────────────────
_batch_buffer = {}   # {sample_id: {"b": ..., "i": ..., "x": ...}}
_batch_index  = 0    # index of the next batch file to write
_done_ids     = set()  # all sample IDs already written to disk

def _init_batch_state():
    """Scan existing batch files on startup to support resuming."""
    global _batch_index, _done_ids
    os.makedirs(BATCHED_ROOT, exist_ok=True)
    existing = sorted(
        f for f in os.listdir(BATCHED_ROOT)
        if f.startswith("batch_") and f.endswith(".json")
    )
    for fname in existing:
        with open(os.path.join(BATCHED_ROOT, fname)) as f:
            _done_ids.update(json.load(f).keys())
    _batch_index = len(existing)
    print(f"  Batch state: {len(existing)} existing files, "
          f"{len(_done_ids)} samples already done.")

def _flush_buffer():
    """Write the current buffer to a new batch file and clear it."""
    global _batch_buffer, _batch_index
    if not _batch_buffer:
        return
    fname = f"batch_{_batch_index:04d}.json"
    path  = os.path.join(BATCHED_ROOT, fname)
    with open(path, "w") as f:
        json.dump(_batch_buffer, f, indent=2)
    print(f"  → Flushed {len(_batch_buffer)} samples → {fname}")
    _batch_buffer = {}
    _batch_index += 1

def add_to_batch(sample_id, tokens, psst_vec, w2t_vec, consensus_vec,
                 agreement_status, intonation_vec, break_idx_vec):
    """Add one sample to the buffer; flush to disk when full."""
    _batch_buffer[sample_id] = {
        "b": {
            "tokens":    tokens,
            "consensus": consensus_vec,
            "psst":      psst_vec,
            "wav2tobi":  w2t_vec,
            "status":    agreement_status,
        },
        "i": {"tokens": tokens, "labels": intonation_vec},
        "x": {"tokens": tokens, "labels": break_idx_vec},
    }
    _done_ids.add(sample_id)
    if len(_batch_buffer) >= FILE_BATCH_SIZE:
        _flush_buffer()

def flush_remaining():
    """Flush any partial buffer at the end of a processing run."""
    _flush_buffer()

def label_files_exist(sample_id):
    """True if this sample has already been written to a batch file."""
    return sample_id in _done_ids

def write_meta():
    """Write a single meta.json to BATCHED_ROOT for provenance."""
    meta = {
        "split":        "peoples_speech/clean",
        "annotators":   [PSST_CKPT, WAV2TOBI_CKPT],
        "consensus_window_words":   CONSENSUS_WINDOW,
        "alignment_tolerance_s":    ALIGNMENT_TOLERANCE_S,
        "f0_window_s":              F0_WINDOW_S,
        "f0_slope_threshold_hz":    F0_SLOPE_HZ,
        "batch_size":               FILE_BATCH_SIZE,
        "schema": {
            "b": "boundary — tokens, consensus, psst, wav2tobi, status",
            "i": "intonation — tokens, labels (0=none 1=rising 2=falling 3=level 4=unclear)",
            "x": "break_idx — tokens, labels (\"3\" | \"4\" | \"\")",
        },
        "created": datetime.now().isoformat(timespec="seconds"),
    }
    with open(os.path.join(BATCHED_ROOT, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)
    print("✓ meta.json written.")

# ── Raw inference caches (unchanged — still per-file) ─────────────
# These are intermediate caches for resuming interrupted runs,
# not training data. Per-file format is fine here.

def load_from_cache(sample_id):
    path = os.path.join(ANNOT_CACHE, f"{sample_id}.json")
    if os.path.exists(path):
        with open(path) as f: return json.load(f)
    return None

def save_to_cache(sample_id, result, retries=3, delay=2.0):
    path = os.path.join(ANNOT_CACHE, f"{sample_id}.json")
    for attempt in range(retries):
        try:
            with open(path, "w") as f:
                json.dump(result, f)
            return
        except OSError as e:
            if attempt < retries - 1:
                print(f"    [cache write failed, retrying in {delay}s: {e}]")
                time.sleep(delay)
            else:
                raise

def load_ctc_cache(sample_id):
    path = os.path.join(CTC_CACHE, f"{sample_id}.json")
    if os.path.exists(path):
        with open(path) as f: return json.load(f).get("mfa_words")
    return None

def save_ctc_cache(sample_id, mfa_words, text="", retries=5, wait=10):
    os.makedirs(CTC_CACHE, exist_ok=True)
    path = os.path.join(CTC_CACHE, f"{sample_id}.json")
    for attempt in range(1, retries + 1):
        try:
            with open(path, "w") as f:
                json.dump({"mfa_words": mfa_words, "text": text}, f)
            return
        except OSError as e:
            if attempt == retries:
                print(f"  ✗ Failed to write cache for {sample_id} after {retries} attempts: {e}")
                return
            print(f"  ⚠ OSError on attempt {attempt}/{retries} for {sample_id} — retrying in {wait}s ...")
            time.sleep(wait)

def load_psst_raw(sample_id):
    path = os.path.join(PSST_CACHE_DIR, f"{sample_id}.json")
    if os.path.exists(path):
        with open(path) as f: return json.load(f).get("psst_raw")
    return None

def save_psst_raw(sample_id, psst_raw):
    os.makedirs(PSST_CACHE_DIR, exist_ok=True)
    with open(os.path.join(PSST_CACHE_DIR, f"{sample_id}.json"), "w") as f:
        json.dump({"sample_id": sample_id, "psst_raw": psst_raw}, f)

def load_w2t_raw(sample_id):
    path = os.path.join(W2T_CACHE_DIR, f"{sample_id}.json")
    if os.path.exists(path):
        with open(path) as f: return json.load(f).get("detections")
    return None

def save_w2t_raw(sample_id, detections):
    os.makedirs(W2T_CACHE_DIR, exist_ok=True)
    with open(os.path.join(W2T_CACHE_DIR, f"{sample_id}.json"), "w") as f:
        json.dump({"sample_id": sample_id, "detections": detections}, f)

# ── Initialise ────────────────────────────────────────────────────
for _d in [ANNOT_CACHE, BATCHED_ROOT, CTC_CACHE, PSST_CACHE_DIR, W2T_CACHE_DIR]:
    os.makedirs(_d, exist_ok=True)

write_meta()
_init_batch_state()
print("✓ Cell 10: Label I/O helpers defined (batch format).")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 11 — Main per-sample pipeline                         ║
# ╚══════════════════════════════════════════════════════════════╝
import soundfile as sf
import os
import time


def process_sample(sample_id, audio_array, sr, mfa_words, text,
                   psst_raw_override=None, w2t_detections_override=None):
    """
    Run the full annotation pipeline on one sample.

    psst_raw_override       : pass a cached PSST string to skip model inference
    w2t_detections_override : pass cached W2T detections to skip model inference

    Per-step timings are printed at the end for profiling.

    Final-word invariants enforced:
      • PSST is forced to fire on the last word (utterance-end is always a
        phrase boundary, linguistic prior).
      • Consensus is forced to True on the last word uniformly, regardless of
        whether W2T corroborated within tolerance. Forced-consensus-without-W2T
        words receive break_idx="4" (IP) as a default, since declarative
        utterance ends are overwhelmingly intonational phrase boundaries.
    """
    t0_total = time.perf_counter()
    print(f"  Processing {sample_id} ({len(mfa_words)} words)...")

    # ── Step 1: Resample ─────────────────────────────────────────
    t0 = time.perf_counter()
    audio16 = librosa.resample(audio_array, orig_sr=sr, target_sr=16000) \
              if sr != 16000 else audio_array.copy()
    t_resample = time.perf_counter() - t0

    # ── Step 2: PSST ─────────────────────────────────────────────
    t0 = time.perf_counter()
    if psst_raw_override is not None:
        psst_raw = psst_raw_override
    else:
        psst_raw = run_psst(audio16)
    psst_indices = psst_to_word_boundaries(psst_raw, mfa_words)

    # Force final-word PSST boundary.
    last_idx = len(mfa_words) - 1
    if last_idx >= 0:
        psst_indices = sorted(set(psst_indices) | {last_idx})

    t_psst = time.perf_counter() - t0
    cache_tag = " [cached]" if psst_raw_override is not None else ""
    print(f"    PSST:     {len(psst_indices)} boundaries → {psst_indices}  "
          f"[{t_psst:.1f}s{cache_tag}]")

    # ── Step 3: Wav2ToBI ─────────────────────────────────────────
    t0 = time.perf_counter()
    if w2t_detections_override is not None:
        w2t_detections = w2t_detections_override
    else:
        w2t_detections = run_wav2tobi(audio16)
    w2t_pairs     = wav2tobi_to_word_boundaries(w2t_detections, mfa_words)
    w2t_indices   = [idx for idx, _ in w2t_pairs]
    w2t_break_map = {idx: bidx for idx, bidx in w2t_pairs}
    t_w2t = time.perf_counter() - t0
    cache_tag = " [cached]" if w2t_detections_override is not None else ""
    print(f"    Wav2ToBI: {len(w2t_detections)} boundaries → {w2t_indices}  "
          f"[{t_w2t:.1f}s{cache_tag}]")

    # ── Step 4: Consensus ─────────────────────────────────────────
    t0 = time.perf_counter()
    consensus_indices = compute_consensus(psst_indices, w2t_pairs)
    agreement_status  = classify_agreement(psst_indices, w2t_indices, len(mfa_words))

    # Force consensus=1 on the last word uniformly. agreement_status stays
    # honest about provenance (it may still say 'psst_only' if W2T did not
    # corroborate), but the training-label vector asserts the linguistic prior.
    if last_idx >= 0:
        consensus_indices = sorted(set(consensus_indices) | {last_idx})

    t_consensus = time.perf_counter() - t0
    print(f"    Consensus: {len(consensus_indices)} boundaries → {consensus_indices}  "
          f"[{t_consensus:.2f}s]")

    # ── Steps 5–6: Per-token label vectors ──
    t0 = time.perf_counter()
    tokens = []
    psst_vec = []; w2t_vec = []; consensus_vec = []
    intonation_vec = []; break_idx_vec = []

    consensus_set = set(consensus_indices)
    psst_set      = set(psst_indices)
    w2t_set       = set(w2t_indices)

    for i, w in enumerate(mfa_words):
        tokens.append(w['word'])
        psst_vec.append(1 if i in psst_set else 0)
        w2t_vec.append(1 if i in w2t_set else 0)
        is_consensus = (i in consensus_set)
        consensus_vec.append(1 if is_consensus else 0)
        if is_consensus:
            inton = classify_intonation(audio16, 16000, w['end'])
            intonation_vec.append(INTONATION_MAP.get(inton, 0))

            # Window-aware break_idx: prefer the W2T detection on the exact
            # word; if absent, scan ±CONSENSUS_WINDOW for a bridged neighbor.
            bi = w2t_break_map.get(i, "")
            if not bi:
                for offset in range(1, CONSENSUS_WINDOW + 1):
                    for cand in (i - offset, i + offset):
                        if cand in w2t_break_map:
                            bi = w2t_break_map[cand]
                            break
                    if bi:
                        break
            # Fallback for forced-consensus utterance-final with no W2T support.
            if not bi and i == last_idx:
                bi = "4"
            break_idx_vec.append(bi or "")
        else:
            intonation_vec.append(0)
            break_idx_vec.append("")
    t_labels = time.perf_counter() - t0

    # ── Step 7: Add to batch buffer ──────────────────────────────
    t0 = time.perf_counter()
    add_to_batch(sample_id, tokens, psst_vec, w2t_vec, consensus_vec,
                 agreement_status, intonation_vec, break_idx_vec)
    t_io = time.perf_counter() - t0

    # ── Save full annotation to raw cache ──
    n_psst_only = sum(1 for s in agreement_status if s == 'psst_only')
    n_w2t_only  = sum(1 for s in agreement_status if s == 'wav2tobi_only')

    result = {
        'sample_id': sample_id, 'text': text, 'psst_raw': psst_raw,
        '_w2t_detections': w2t_detections,
        'words': [
            {
                'word':               mfa_words[i]['word'],
                'start_s':            round(mfa_words[i]['start'], 4),
                'end_s':              round(mfa_words[i]['end'],   4),
                'psst_boundary':      bool(psst_vec[i]),
                'wav2tobi_boundary':  bool(w2t_vec[i]),
                'consensus_boundary': bool(consensus_vec[i]),
                'break_idx':          break_idx_vec[i] or None,
                'intonation':         {v: k for k, v in INTONATION_MAP.items()}[intonation_vec[i]],
                'agreement_status':   agreement_status[i],
            }
            for i in range(len(mfa_words))
        ],
        'meta': {
            'psst_n_boundaries':      len(psst_indices),
            'wav2tobi_n_boundaries':  len(w2t_detections),
            'consensus_n_boundaries': len(consensus_indices),
            'n_psst_only':            n_psst_only,
            'n_wav2tobi_only':        n_w2t_only,
            'agreement_rate': round(len(consensus_indices) / max(len(psst_indices), 1), 3),
            'patch_version':          2,
        }
    }
    save_to_cache(sample_id, result)

    t_total = time.perf_counter() - t0_total
    print(f"    ⏱  resample={t_resample:.2f}s  psst={t_psst:.2f}s  "
          f"w2t={t_w2t:.2f}s  consensus={t_consensus:.2f}s  "
          f"labels={t_labels:.2f}s  io={t_io:.2f}s  │  total={t_total:.2f}s")
    return result


print("✓ Cell 11: Main pipeline function defined (with per-step profiling).")

---
## Section 5 · Data Loading

Streams `MLCommons/peoples_speech` (clean) and runs CTC forced alignment
for word timestamps. Audio is already at 16 kHz — resampling is skipped
when `sampling_rate == 16000`.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 12 — Load People's Speech + CTC forced alignment       ║
# ║                                                              ║
# ║  Mirrors the LibriTTS Cell 12 exactly, with three changes:   ║
# ║    1. Dataset: MLCommons/peoples_speech, config "clean"      ║
# ║    2. Text field: "text" (not "text_normalized")             ║
# ║    3. Audio is already 16 kHz — resampling is skipped when   ║
# ║       sampling_rate == 16000 (always true for clean subset,  ║
# ║       but checked defensively).                              ║
# ║                                                              ║
# ║  Sample selection uses master_ids (built in Cell 2c) so      ║
# ║  both sessions work through the same ordered list at their   ║
# ║  own pace. No coordination or batch locking needed.          ║
# ╚══════════════════════════════════════════════════════════════╝
import re, torch, torchaudio
import numpy as np
import librosa
import soundfile as sf
from datasets import load_dataset


# ── CTC alignment helpers (identical to LibriTTS pipeline) ───────────────────
def text_to_tokens(text, labels):
    label2idx = {l: i for i, l in enumerate(labels)}
    tokens = []
    for word in text.upper().split():
        for char in word:
            if char in label2idx:
                tokens.append(label2idx[char])
        tokens.append(label2idx['|'])
    return tokens

def get_trellis(emission, tokens, blank_id=0):
    T, V = emission.shape
    N = len(tokens)
    trellis = torch.full((T + 1, N + 1), -float("inf"))
    trellis[0, 0] = 0.0
    for t in range(T):
        trellis[t+1, 1:] = torch.maximum(
            trellis[t, 1:]  + emission[t, blank_id],
            trellis[t, :-1] + emission[t, tokens],
        )
        trellis[t+1, 1:] = torch.maximum(
            trellis[t+1, 1:],
            trellis[t, 1:] + emission[t, tokens],
        )
    return trellis

def backtrack(trellis, emission, tokens, blank_id=0):
    t = trellis.size(0) - 1
    j = trellis.size(1) - 1
    path = [(j - 1, t - 1)]
    while j > 1:
        prob_stay    = trellis[t-1, j]   + emission[t-1, blank_id]
        prob_advance = trellis[t-1, j-1] + emission[t-1, tokens[j-2]]
        if prob_advance > prob_stay:
            j -= 1
        t -= 1
        path.append((j - 1, t - 1))
    return path[::-1]

def merge_to_words(path, labels, tokens, stride_s, original_words=None):
    words = []
    current_chars = []
    word_start_frame = None
    for tok_idx, frame in path:
        label = labels[tokens[tok_idx]]
        if label == '|':
            if current_chars:
                word_idx = len(words)
                word = (original_words[word_idx]
                        if original_words and word_idx < len(original_words)
                        else ''.join(current_chars))
                words.append({
                    'word':  word,
                    'start': round(word_start_frame * stride_s, 4),
                    'end':   round(frame * stride_s, 4),
                })
                current_chars = []
                word_start_frame = None
        else:
            if not current_chars:
                word_start_frame = frame
            current_chars.append(label)
    if current_chars:
        word_idx = len(words)
        word = (original_words[word_idx]
                if original_words and word_idx < len(original_words)
                else ''.join(current_chars))
        words.append({
            'word':  word,
            'start': round(word_start_frame * stride_s, 4),
            'end':   round(path[-1][1] * stride_s, 4),
        })
    return words

def get_word_timestamps(audio16k, text):
    text_clean = re.sub(r"[^a-zA-Z\s']", '', text).strip()
    if not text_clean:
        return []
    orig_words  = text.split()
    clean_words = text_clean.split()
    if len(orig_words) != len(clean_words):
        print(f"    ⚠ word-count mismatch: raw={len(orig_words)} "
              f"clean={len(clean_words)} — skipping alignment")
        return []
    try:
        waveform = torch.tensor(audio16k, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            emission, _ = fa_model(waveform)
        emission = torch.log_softmax(emission[0], dim=-1).cpu()
        tokens   = text_to_tokens(text_clean, fa_labels)
        if not tokens:
            return []
        trellis = get_trellis(emission, tokens)
        path    = backtrack(trellis, emission, tokens)
        return merge_to_words(path, fa_labels, tokens, FA_STRIDE_S,
                              original_words=orig_words)
    except Exception as e:
        print(f"    ⚠ CTC alignment failed: {e}")
        return []


# ── Select next N_SAMPLES from master ID list ────────────────────────────
print(f"Selecting up to {N_SAMPLES} samples from master ID list...")
print(f"  Master list: {len(master_ids)} total  |  Already done: {len(completed_ids)}")

target_ids = []
for sid in master_ids:
    if sid in completed_ids:
        continue
    target_ids.append(sid)
    if len(target_ids) >= N_SAMPLES:
        break

print(f"  → {len(target_ids)} samples to process this run.")
if target_ids:
    print(f"  First: {target_ids[0]}   Last: {target_ids[-1]}")

target_set = set(target_ids)

# ── Stream People's Speech, fetching only the selected samples ────
# People's Speech "clean" split uses the "train" split name.
# Each sample: id (str), audio (dict), duration_ms (float), text (str)
# Audio is already at 16 kHz in the clean subset (MLCommons spec),
# but we verify sampling_rate and resample defensively if needed.
print(f"\nStreaming MLCommons/peoples_speech (clean) to fetch selected samples...")
ds = load_dataset(
    "MLCommons/peoples_speech",
    "clean",
    split="train",
    streaming=True,
    trust_remote_code=True,
)

raw_samples = {}
for sample in ds:
    sid = sample['id']
    if sid in target_set:
        raw_samples[sid] = sample
        if len(raw_samples) >= len(target_ids):
            break

raw_samples_ordered = [raw_samples[sid] for sid in target_ids if sid in raw_samples]
missing = [sid for sid in target_ids if sid not in raw_samples]
if missing:
    print(f"  ⚠ {len(missing)} samples not found in stream: {missing[:5]}")
print(f"  ✓ Fetched {len(raw_samples_ordered)} samples.")


# ── CTC alignment (with disk cache) ──────────────────────────────────
print("\nRunning CTC forced alignment (checking ctc_cache first)...")
normalized_samples = []

for i, s in enumerate(raw_samples_ordered):
    sid  = s['id']
    text = s['text']         # People's Speech uses 'text', not 'text_normalized'

    # ── Already fully annotated ────────────────────────────────────────────
    if not FORCE_REPROCESS and label_files_exist(sid):
        cached = load_from_cache(sid)
        if cached:
            print(f"[{i+1}/{len(raw_samples_ordered)}] {sid} — labels exist, skipping")
            mfa_words = [
                {'word': w['word'], 'start': w['start_s'], 'end': w['end_s']}
                for w in cached['words']
            ]
            normalized_samples.append({
                'id': sid, 'text': text,
                'audio': s['audio'],
                'mfa_words': mfa_words, '_cached': True,
            })
            continue

    # ── CTC cache hit ─────────────────────────────────────────────────────
    mfa_words = load_ctc_cache(sid)
    if mfa_words:
        print(f"[{i+1}/{len(raw_samples_ordered)}] {sid} — CTC cache hit ({len(mfa_words)} words)")
        normalized_samples.append({
            'id': sid, 'text': text,
            'audio': s['audio'],
            'mfa_words': mfa_words, '_cached': False,
        })
        continue

    # ── merge_only: skip if no CTC cache ─────────────────────────────────
    if RUN_MODE == "merge_only":
        print(f"[{i+1}/{len(raw_samples_ordered)}] {sid} — "
              f"⚠ no CTC cache; run psst_only or full first")
        continue

    # ── Run CTC and save to cache ───────────────────────────────────────────
    audio_arr = np.array(s['audio']['array'])
    sr        = s['audio']['sampling_rate']
    # People's Speech clean is 16 kHz but we check defensively
    audio16   = librosa.resample(audio_arr, orig_sr=sr, target_sr=16000) \
                if sr != 16000 else audio_arr

    if KEEP_AUDIO:
        wav_path = os.path.join(AUDIO_CACHE, f"{sid}.wav")
        if not os.path.exists(wav_path):
            sf.write(wav_path, audio16, 16000)

    mfa_words = get_word_timestamps(audio16, text)
    if not mfa_words:
        print(f"[{i+1}/{len(raw_samples_ordered)}] {sid} — ⚠ CTC alignment failed, skipping")
        continue

    save_ctc_cache(sid, mfa_words, text)
    print(f"[{i+1}/{len(raw_samples_ordered)}] {sid} — "
          f"{len(mfa_words)} words aligned (saved to ctc_cache)")
    normalized_samples.append({
        'id': sid, 'text': text,
        'audio': s['audio'],
        'mfa_words': mfa_words, '_cached': False,
    })

print(f"\n✓ {len(normalized_samples)} samples ready "
      f"({sum(1 for s in normalized_samples if not s.get('_cached'))} newly aligned).")

if normalized_samples:
    ex = normalized_samples[0]
    print(f"\nSpot-check: {ex['id']}")
    print(f"  Text:  {ex['text'][:70]}")
    print(f"  Words: {ex['mfa_words'][:3]}")

---
## Section 6 · Run Pipeline

Iterates over the selected samples and runs the annotation pipeline.
Behaviour is controlled by `RUN_MODE` (set in Cell 1).

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 13 — Main processing loop                             ║
# ║                                                              ║
# ║  Behaviour is controlled by RUN_MODE (set in Cell 1):       ║
# ║                                                              ║
# ║  "full"       → batched PSST + per-sample W2T + merge       ║
# ║  "psst_only"  → batched PSST only; saves psst_cache         ║
# ║  "w2t_only"   → per-sample Wav2ToBI only; saves w2t_cache   ║
# ║  "merge_only" → reads all caches; writes batch label files  ║
# ║                                                              ║
# ║  KEY: full mode batches PSST across PSST_BATCH_SIZE samples ║
# ║  before doing per-sample W2T+merge. This is what gives the  ║
# ║  ~90x speedup vs calling process_sample() one at a time.    ║
# ╚══════════════════════════════════════════════════════════════╝
import soundfile as sf
import traceback
import time

results  = []
n_fresh  = 0
n_cached = 0
n_failed = 0

print(f"Processing {len(normalized_samples)} samples  [RUN_MODE='{RUN_MODE}']")
print(f"Label output: {BATCHED_ROOT}")
print(f"Force reprocess: {FORCE_REPROCESS}")
print()

# ══════════════════════════════════════════════════════════════════
# PSST-ONLY PASS  (batched)
# Saves: psst_cache
# ══════════════════════════════════════════════════════════════════
if RUN_MODE == "psst_only":
    pending = [s for s in normalized_samples
               if not s.get('_cached') and load_psst_raw(s['id']) is None]
    print(f"PSST pass: {len(pending)} samples to annotate "
          f"(batch_size={PSST_BATCH_SIZE})")

    for batch_start in range(0, len(pending), PSST_BATCH_SIZE):
        batch = pending[batch_start: batch_start + PSST_BATCH_SIZE]
        ids   = [s['id'] for s in batch]

        audio_arrs = []
        for s in batch:
            arr = np.array(s['audio']['array'])
            sr  = s['audio']['sampling_rate']
            if sr != 16000:
                arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
            audio_arrs.append(arr)

        t0 = time.perf_counter()
        raw_outputs = run_psst_batch(audio_arrs)
        elapsed = time.perf_counter() - t0

        for s, raw in zip(batch, raw_outputs):
            save_psst_raw(s['id'], raw)
            n_fresh += 1

        end_idx = min(batch_start + PSST_BATCH_SIZE, len(pending))
        print(f"  [{batch_start+1}–{end_idx}/{len(pending)}]  "
              f"{elapsed:.1f}s total  ({elapsed/len(batch):.1f}s/sample)  ids={ids}")

    already_done = len(normalized_samples) - len(pending)
    flush_remaining()
    print(f"\n✓ PSST pass complete.")
    print(f"  Newly annotated: {n_fresh}   Already cached: {already_done}")
    print(f"  Outputs: {PSST_CACHE_DIR}")

# ══════════════════════════════════════════════════════════════════
# WAV2TOBI-ONLY PASS
# Reads: ctc_cache  |  Saves: w2t_cache
# ══════════════════════════════════════════════════════════════════
elif RUN_MODE == "w2t_only":
    pending = [s for s in normalized_samples
               if not s.get('_cached') and load_w2t_raw(s['id']) is None]
    print(f"Wav2ToBI pass: {len(pending)} samples to annotate")

    for i, s in enumerate(pending):
        sid = s['id']
        print(f"[{i+1}/{len(pending)}] {sid}")
        try:
            arr = np.array(s['audio']['array'])
            sr  = s['audio']['sampling_rate']
            a16 = librosa.resample(arr, orig_sr=sr, target_sr=16000) \
                  if sr != 16000 else arr
            t0 = time.perf_counter()
            detections = run_wav2tobi(a16)
            elapsed = time.perf_counter() - t0
            save_w2t_raw(sid, detections)
            print(f"  ✓ {len(detections)} boundaries  [{elapsed:.1f}s]")
            n_fresh += 1
        except Exception as e:
            print(f"  ✗ FAILED: {e}")
            traceback.print_exc()
            n_failed += 1
        print()
    flush_remaining()
    print(f"✓ Wav2ToBI pass complete.  Annotated: {n_fresh}   Failed: {n_failed}")
    print(f"  Outputs: {W2T_CACHE_DIR}")

# ══════════════════════════════════════════════════════════════════
# MERGE-ONLY PASS  (no GPU needed)
# Reads: psst_cache + w2t_cache + ctc_cache
# Writes: labels/{split_tag}/
# ══════════════════════════════════════════════════════════════════
elif RUN_MODE == "merge_only":
    pending = [s for s in normalized_samples
               if not s.get('_cached') and not label_files_exist(s['id'])]
    print(f"Merge pass: {len(pending)} samples to merge")

    for i, s in enumerate(pending):
        sid       = s['id']
        mfa_words = s['mfa_words']
        text      = s['text']
        print(f"[{i+1}/{len(pending)}] {sid}")

        psst_raw = load_psst_raw(sid)
        w2t_dets = load_w2t_raw(sid)

        if psst_raw is None:
            print(f"  ⚠ No PSST cache — run psst_only mode first.  Skipping.")
            n_failed += 1
            continue
        if w2t_dets is None:
            print(f"  ⚠ No W2T cache  — run w2t_only mode first.  Skipping.")
            n_failed += 1
            continue

        try:
            arr = np.array(s['audio']['array'])
            sr  = s['audio']['sampling_rate']
            result = process_sample(sid, arr, sr, mfa_words, text,
                                    psst_raw_override=psst_raw,
                                    w2t_detections_override=w2t_dets)
            results.append(result)
            n_fresh += 1

            completed_ids.add(sid)
            with open(PROGRESS_FILE, 'w') as f:
                json.dump(list(completed_ids), f)

            m = result['meta']
            print(f"  ✓ PSST: {m['psst_n_boundaries']}  "
                  f"W2T: {m['wav2tobi_n_boundaries']}  "
                  f"Consensus: {m['consensus_n_boundaries']}  "
                  f"Agreement: {m['agreement_rate']:.2f}")
        except Exception as e:
            print(f"  ✗ FAILED: {e}")
            traceback.print_exc()
            n_failed += 1
        print()
    flush_remaining()
    print("=" * 60)
    print(f"  Merged: {n_fresh}    Failed: {n_failed}")
    print(f"  Labels written to: {BATCHED_ROOT}")
    print("=" * 60)

# ══════════════════════════════════════════════════════════════════
# FULL PASS — batched PSST + per-sample W2T + merge
#
# Structure:
#   1. Collect a batch of PSST_BATCH_SIZE pending samples
#   2. For each sample in the batch, check psst_cache first
#   3. Run run_psst_batch() only on the uncached ones
#   4. For each sample: run W2T (or load from w2t_cache), merge, save
#
# This means PSST runs at batch throughput (~4 samples per forward
# pass) while W2T and merge still happen per-sample. Result: the
# same ~90x speedup as psst_only mode, but in a single session.
# ══════════════════════════════════════════════════════════════════
else:
    pending = [s for s in normalized_samples if not s.get('_cached')]
    print(f"Full pass: {len(pending)} samples to annotate "
          f"(PSST batch_size={PSST_BATCH_SIZE})")
    print()

    for batch_start in range(0, len(pending), PSST_BATCH_SIZE):
        batch = pending[batch_start: batch_start + PSST_BATCH_SIZE]

        # ── Load audio for this batch ─────────────────────────────
        audio_arrs = []
        srs        = []
        for s in batch:
            arr = np.array(s['audio']['array'])
            sr  = s['audio']['sampling_rate']
            audio_arrs.append(arr)
            srs.append(sr)

        # ── PSST: check cache, batch-infer only what's needed ─────
        psst_raws        = [load_psst_raw(s['id']) for s in batch]
        needs_psst       = [j for j, r in enumerate(psst_raws) if r is None]

        if needs_psst:
            t0 = time.perf_counter()
            uncached_arrs   = [audio_arrs[j] for j in needs_psst]
            uncached_srs    = [srs[j]        for j in needs_psst]
            batch_psst_out  = run_psst_batch(uncached_arrs, sr=16000)
            t_psst_batch    = time.perf_counter() - t0
            for j, raw in zip(needs_psst, batch_psst_out):
                psst_raws[j] = raw
                save_psst_raw(batch[j]['id'], raw)
            print(f"  PSST batch [{batch_start+1}–"
                  f"{min(batch_start+PSST_BATCH_SIZE, len(pending))}/{len(pending)}]: "
                  f"{len(needs_psst)} inferred in {t_psst_batch:.1f}s "
                  f"({t_psst_batch/len(needs_psst):.2f}s/sample), "
                  f"{len(batch)-len(needs_psst)} from cache")

        # ── Per-sample: W2T + merge ───────────────────────────────
        for s, arr, sr, psst_raw in zip(batch, audio_arrs, srs, psst_raws):
            sid = s['id']
            print(f"[{batch_start + batch.index(s) + 1}/{len(pending)}] {sid}")
            try:
                w2t_cached = load_w2t_raw(sid)

                result = process_sample(
                    sid, arr, sr, s['mfa_words'], s['text'],
                    psst_raw_override=psst_raw,
                    w2t_detections_override=w2t_cached,
                )
                results.append(result)
                n_fresh += 1

                # Persist W2T cache for resumability
                if w2t_cached is None:
                    save_w2t_raw(sid, result['_w2t_detections'])

                completed_ids.add(sid)
                with open(PROGRESS_FILE, 'w') as f:
                    json.dump(list(completed_ids), f)

                m = result['meta']
                print(f"  ✓ PSST: {m['psst_n_boundaries']}  "
                      f"W2T: {m['wav2tobi_n_boundaries']}  "
                      f"Consensus: {m['consensus_n_boundaries']}  "
                      f"(+{m['n_psst_only']} PSST-only, "
                      f"+{m['n_wav2tobi_only']} W2T-only)  "
                      f"Agreement: {m['agreement_rate']:.2f}")

            except Exception as e:
                print(f"  ✗ FAILED: {e}")
                traceback.print_exc()
                n_failed += 1
            print()

    # Handle cached samples (already annotated — just load result)
    for sample in normalized_samples:
        if not sample.get('_cached'):
            continue
        cached = load_from_cache(sample['id'])
        if cached:
            results.append(cached)
            n_cached += 1

    flush_remaining()
    print("=" * 60)
    print(f"  Fresh:  {n_fresh}    Cached: {n_cached}    Failed: {n_failed}")
    print(f"  Total:  {len(results)} samples annotated")
    print(f"  Labels written to: {BATCHED_ROOT}")
    print("=" * 60)


---
## Section 7 · Analysis

Summary statistics across all processed samples.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 14 — Summary statistics                               ║
# ╚══════════════════════════════════════════════════════════════╝
import json, os
import numpy as np

# ── Load all batch label files ──────────────────────────────────────
all_samples = []
batch_files = sorted(
    f for f in os.listdir(BATCHED_ROOT)
    if f.startswith("batch_") and f.endswith(".json")
)

if not batch_files:
    print("No batch files found. Run Cell 13 first.")
else:
    print(f"Loading {len(batch_files)} batch files from {BATCHED_ROOT}...")
    for fname in batch_files:
        with open(os.path.join(BATCHED_ROOT, fname)) as f:
            batch = json.load(f)
        for sid, data in batch.items():
            b      = data.get("b", {})
            i_data = data.get("i", {})
            x_data = data.get("x", {})
            status    = b.get("status", [])
            consensus = b.get("consensus", [])
            psst_v    = b.get("psst", [])
            w2t_v     = b.get("wav2tobi", [])
            inton_lbl = i_data.get("labels", []) if i_data else []
            bidx_lbl  = x_data.get("labels", []) if x_data else []
            all_samples.append({
                'id':               sid,
                'n_tokens':         len(consensus),
                'n_consensus':      sum(consensus),
                'n_psst_only':      sum(1 for s in status if s == 'psst_only'),
                'n_w2t_only':       sum(1 for s in status if s == 'wav2tobi_only'),
                'n_psst':           sum(psst_v),
                'n_w2t':            sum(w2t_v),
                'inton_labels':     inton_lbl,
                'bidx_labels':      bidx_lbl,
                'consensus_labels': consensus,
            })

n = len(all_samples)
if n == 0:
    print("No samples loaded.")
else:
    total_words      = sum(s['n_tokens']    for s in all_samples)
    total_consensus  = sum(s['n_consensus'] for s in all_samples)
    total_psst_only  = sum(s['n_psst_only'] for s in all_samples)
    total_w2t_only   = sum(s['n_w2t_only']  for s in all_samples)
    total_psst       = sum(s['n_psst']      for s in all_samples)
    total_w2t        = sum(s['n_w2t']       for s in all_samples)

    agree_rates = [
        s['n_consensus'] / max(s['n_psst'], 1)
        for s in all_samples
    ]

    inton_map = {0:'none', 1:'rising', 2:'falling', 3:'level', 4:'unclear'}
    inton_counts = {k: 0 for k in inton_map}
    for s in all_samples:
        for bl, il in zip(s['consensus_labels'], s['inton_labels']):
            if bl == 1:
                inton_counts[il] += 1

    bidx_counts = {'3': 0, '4': 0}
    for s in all_samples:
        for bl, bi in zip(s['consensus_labels'], s['bidx_labels']):
            if bl == 1 and bi in bidx_counts:
                bidx_counts[bi] += 1

    SEP = "=" * 64
    print(SEP)
    print("  ANNOTATION SUMMARY — People's Speech (clean)")
    print(SEP)
    print(f"  Samples:                  {n}")
    print(f"  Total words:              {total_words}")
    print()
    print("  Boundary counts:")
    print(f"    PSST total:             {total_psst}")
    print(f"    Wav2ToBI total:         {total_w2t}")
    print(f"    Consensus (agreed):     {total_consensus}  ({100*total_consensus/max(total_words,1):.1f}% of words)")
    print(f"    PSST-only:              {total_psst_only}")
    print(f"    Wav2ToBI-only:          {total_w2t_only}")
    print()
    print("  Agreement (consensus / PSST boundaries):")
    print(f"    Mean:  {np.mean(agree_rates):.3f}")
    print(f"    Std:   {np.std(agree_rates):.3f}")
    print(f"    Range: {min(agree_rates):.3f} – {max(agree_rates):.3f}")
    print()
    print("  Intonation at consensus boundaries:")
    for code, label in inton_map.items():
        if code == 0: continue
        count = inton_counts[code]
        pct   = 100 * count / max(total_consensus, 1)
        bar   = '█' * int(pct / 2)
        print(f"    {label:10s}: {count:4d}  ({pct:5.1f}%)  {bar}")
    print()
    print("  Break index at consensus boundaries:")
    for bidx, label in [('3','intermediate phrase'), ('4','intonational phrase')]:
        count = bidx_counts[bidx]
        pct   = 100 * count / max(total_consensus, 1)
        bar   = '█' * int(pct / 2)
        print(f"    {bidx} ({label:22s}): {count:4d}  ({pct:5.1f}%)  {bar}")
    print()
    print("  Per-sample breakdown (first 20):")
    print(f"  {'Sample ID':<40} {'Tok':>4} {'PSST':>5} {'W2T':>5} {'Cons':>5} {'Agree':>6}")
    print(f"  {'-'*40} {'-'*4} {'-'*5} {'-'*5} {'-'*5} {'-'*6}")
    for s in all_samples[:20]:
        ar = s['n_consensus'] / max(s['n_psst'], 1)
        print(f"  {s['id']:<40} {s['n_tokens']:>4} {s['n_psst']:>5} "
              f"{s['n_w2t']:>5} {s['n_consensus']:>5} {ar:>6.2f}")
    if n > 20:
        print(f"  ... ({n - 20} more samples)")
    print(SEP)
    print(f"  Label files: {BATCHED_ROOT}")
    print(SEP)

In [ ]:

import json, os, glob

# ── Paths (copy from Cell 1) ──────────────────────────────────────
DRIVE_ROOT   = "/content/drive/MyDrive/Capstone/project"
CACHE_ROOT   = f"{DRIVE_ROOT}/cache/peoples_speech"

MASTER_ID_FILE = f"{CACHE_ROOT}/master_ids_peoples_speech.json"
PROGRESS_FILE  = f"{CACHE_ROOT}/progress_peoples_speech.json"
BATCHED_ROOT   = f"{DRIVE_ROOT}/labels/ps"

# ── Load master ID list ───────────────────────────────────────────
if not os.path.exists(MASTER_ID_FILE):
    print(f"✗ master_ids file not found: {MASTER_ID_FILE}")
    master_ids = []
else:
    with open(MASTER_ID_FILE) as f:
        master_ids = json.load(f)
    print(f"✓ Master IDs loaded: {len(master_ids):,}")

# ── Load progress file ────────────────────────────────────────────
if not os.path.exists(PROGRESS_FILE):
    print(f"✗ Progress file not found: {PROGRESS_FILE}")
    completed_ids = set()
else:
    with open(PROGRESS_FILE) as f:
        progress_data = json.load(f)
    # progress file is a list or a dict with an "annotated" key — handle both
    if isinstance(progress_data, list):
        completed_ids = set(progress_data)
    elif isinstance(progress_data, dict):
        completed_ids = set(progress_data.get("annotated", progress_data.keys()))
    print(f"✓ Progress file loaded: {len(completed_ids):,} completed IDs")

# ── Cross-check against batch files ──────────────────────────────
batch_files = sorted(glob.glob(os.path.join(BATCHED_ROOT, "batch_*.json")))
ids_in_batches = set()
for bf in batch_files:
    with open(bf) as f:
        batch = json.load(f)
    ids_in_batches.update(batch.keys())

# ── Summary ───────────────────────────────────────────────────────
total        = len(master_ids)
done_progress = len(completed_ids)
done_batches  = len(ids_in_batches)
remaining     = total - done_progress

print()
print("=" * 50)
print("  PEOPLE'S SPEECH ANNOTATION PROGRESS")
print("=" * 50)
print(f"  Total (master list) : {total:>10,}")
print(f"  Completed (progress): {done_progress:>10,}  ({done_progress/total*100:.1f}% of total)" if total else "  Completed (progress):          —")
print(f"  IDs in batch files  : {done_batches:>10,}")
print(f"  Remaining           : {remaining:>10,}")
print(f"  Batch files found   : {len(batch_files):>10,}")

if done_progress != done_batches:
    print()
    print(f"  ⚠ Mismatch: progress file says {done_progress:,} done "
          f"but batch files contain {done_batches:,} IDs.")
    print(f"    Difference: {abs(done_progress - done_batches):,} IDs")

print("=" * 50)

In [ ]:
# run when there's a mismatch (trims completed ID's that don't appear)
'''
import json, os, glob, shutil
from datetime import datetime

# ── Paths (copy from Cell 1) ──────────────────────────────────────
DRIVE_ROOT   = "/content/drive/MyDrive/Capstone/project"
CACHE_ROOT   = f"{DRIVE_ROOT}/cache/peoples_speech"

PROGRESS_FILE  = f"{CACHE_ROOT}/progress_peoples_speech.json"
BATCHED_ROOT   = f"{DRIVE_ROOT}/labels/ps"

# ── Load progress file ────────────────────────────────────────────
with open(PROGRESS_FILE) as f:
    progress_data = json.load(f)

if isinstance(progress_data, list):
    completed_ids = set(progress_data)
elif isinstance(progress_data, dict):
    completed_ids = set(progress_data.get("annotated", progress_data.keys()))
else:
    raise ValueError(f"Unexpected progress file format: {type(progress_data)}")

print(f"✓ Progress file loaded: {len(completed_ids):,} IDs")

# ── Collect IDs present in batch files ───────────────────────────
batch_files = sorted(glob.glob(os.path.join(BATCHED_ROOT, "batch_*.json")))
ids_in_batches = set()
for bf in batch_files:
    with open(bf) as f:
        batch = json.load(f)
    ids_in_batches.update(batch.keys())

print(f"✓ Batch files scanned: {len(ids_in_batches):,} IDs across {len(batch_files)} files")

# ── Find orphaned IDs ─────────────────────────────────────────────
orphaned = completed_ids - ids_in_batches
print(f"\n  Orphaned IDs (in progress, missing from batches): {len(orphaned):,}")

if not orphaned:
    print("  Nothing to fix — progress file and batch files are in sync.")
else:
    # ── Backup first ──────────────────────────────────────────────
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = PROGRESS_FILE.replace(".json", f"_backup_{timestamp}.json")
    shutil.copy2(PROGRESS_FILE, backup_path)
    print(f"  ✓ Backup saved → {backup_path}")

    # ── Write corrected progress file ─────────────────────────────
    cleaned_ids = completed_ids - orphaned

    if isinstance(progress_data, list):
        cleaned = sorted(cleaned_ids)
    elif isinstance(progress_data, dict) and "annotated" in progress_data:
        cleaned = dict(progress_data)
        cleaned["annotated"] = sorted(cleaned_ids)
    else:
        # dict keyed by ID — drop the orphaned keys
        cleaned = {k: v for k, v in progress_data.items() if k not in orphaned}

    with open(PROGRESS_FILE, "w") as f:
        json.dump(cleaned, f)

    print(f"  ✓ Progress file updated: {len(cleaned_ids):,} IDs retained, {len(orphaned):,} dropped")
    print(f"\n  Pipeline will reprocess {len(orphaned):,} IDs on next run.")

In [ ]:
import os
cache_dir = "/content/drive/MyDrive/Capstone/project/cache/peoples_speech/ctc/"
n_cached = len(os.listdir(cache_dir))
print(f"CTC cache: {n_cached:,} files")